# 前馈神经网络（上）激活函数 + 两层 MLP 二分类

本节做三件事：

1. 看一下常用**激活函数**的形状（sigmoid / tanh / ReLU 族）。
2. 用 PyTorch tensor 手写一个两层 MLP 的**前向 + 反向**，再用 `autograd` 跑一遍同一个网络，验证梯度逐元素一致——感受 autograd 在做什么。
3. 用 `nn.Module` 在 moons 数据集上完成二分类，画决策边界。

## 1. 激活函数

PyTorch 把激活函数放在 `torch.nn.functional` 里（也有 `nn.ReLU` 等 module 形式）。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)

x = torch.linspace(-6, 6, 400)
acts = {
    'sigmoid':     torch.sigmoid(x),
    'tanh':        torch.tanh(x),
    'relu':        F.relu(x),
    'leaky_relu':  F.leaky_relu(x, negative_slope=0.1),
    'elu':         F.elu(x),
    'softplus':    F.softplus(x),
}

plt.figure(figsize=(9, 5))
for name, y in acts.items():
    plt.plot(x.numpy(), y.numpy(), label=name)
plt.axhline(0, c='gray', lw=0.5); plt.axvline(0, c='gray', lw=0.5)
plt.legend(); plt.title('activation functions'); plt.grid(alpha=0.3); plt.show()

### 激活函数的导数

训练时反向传播需要激活函数的导数。`autograd` 帮我们算，但提前对照一下能加深理解。

In [ ]:
x = torch.linspace(-6, 6, 400, requires_grad=True)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, fn) in zip(axes.flat, [
    ('sigmoid', torch.sigmoid), ('tanh', torch.tanh),
    ('relu', F.relu), ('leaky_relu', lambda t: F.leaky_relu(t, 0.1)),
    ('elu', F.elu), ('softplus', F.softplus),
]):
    y = fn(x)
    grad = torch.autograd.grad(y.sum(), x, create_graph=False)[0]
    ax.plot(x.detach().numpy(), y.detach().numpy(), label=name)
    ax.plot(x.detach().numpy(), grad.detach().numpy(), '--', label=f"d{name}/dx")
    ax.axhline(0, c='gray', lw=0.5); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. 两层 MLP：手算反向 vs autograd

结构：$\boldsymbol{x} \xrightarrow{W_1, b_1} \boldsymbol{z}_1 \xrightarrow{\text{Logistic}} \boldsymbol{a}_1 \xrightarrow{W_2, b_2} z_2 \xrightarrow{\text{Logistic}} \hat y$。

二分类交叉熵损失：$\mathcal{L} = -[y \log \hat y + (1-y) \log (1-\hat y)]$。

对单个样本逐步手算 $\partial \mathcal{L} / \partial W_1, b_1, W_2, b_2$，然后让 autograd 算同一组梯度，验证逐元素一致。

In [ ]:
torch.manual_seed(0)

# 一个样本
x = torch.randn(3)
y = torch.tensor(1.0)

# 参数（先 detach，因为我们要手算，autograd 部分再单独跑）
W1 = torch.randn(4, 3); b1 = torch.zeros(4)
W2 = torch.randn(4);    b2 = torch.zeros(())

# ---- 前向 ----
z1 = W1 @ x + b1
a1 = torch.sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = torch.sigmoid(z2)
loss = -(y * torch.log(a2) + (1 - y) * torch.log(1 - a2))
print(f'forward:  z1.shape={list(z1.shape)}  a2={a2.item():.4f}  loss={loss.item():.4f}')

# ---- 手算反向（链式法则） ----
# dL/da2 = -(y/a2 - (1-y)/(1-a2));  da2/dz2 = a2*(1-a2)
# 化简后：dL/dz2 = a2 - y
dz2 = a2 - y
dW2 = dz2 * a1
db2 = dz2
# 反传到 a1: dL/da1 = dz2 * W2
da1 = dz2 * W2
# 通过 sigmoid: dL/dz1 = da1 * a1*(1-a1)
dz1 = da1 * a1 * (1 - a1)
# 反传到 W1, b1
dW1 = dz1.unsqueeze(1) * x.unsqueeze(0)   # 外积
db1 = dz1
print(f'manual:   dW1.shape={list(dW1.shape)}  dW2.shape={list(dW2.shape)}')

In [ ]:
# ---- autograd 版 ----
W1a = W1.clone().requires_grad_()
b1a = b1.clone().requires_grad_()
W2a = W2.clone().requires_grad_()
b2a = b2.clone().requires_grad_()

z1a = W1a @ x + b1a
a1a = torch.sigmoid(z1a)
z2a = W2a @ a1a + b2a
a2a = torch.sigmoid(z2a)
loss_a = F.binary_cross_entropy(a2a, y)
loss_a.backward()

print('--- 梯度对比（手算 vs autograd 的最大绝对差） ---')
print(f'dW1:  {(dW1 - W1a.grad).abs().max().item():.2e}')
print(f'db1:  {(db1 - b1a.grad).abs().max().item():.2e}')
print(f'dW2:  {(dW2 - W2a.grad).abs().max().item():.2e}')
print(f'db2:  {(db2 - b2a.grad).abs().max().item():.2e}')

梯度应当在 $10^{-7}$ 量级一致——确认手算公式 $\frac{\partial \mathcal{L}}{\partial z_2} = \hat y - y$ 等推导无误。后续直接信任 `autograd`，不再手算。

## 3. moons 数据集上的两层 MLP

两个交错的半月形——线性分类器分不开，最小可以用一个有少量隐藏神经元的 MLP 解决。

In [ ]:
def make_moons(n=400, noise=0.15, seed=0):
    g = torch.Generator().manual_seed(seed)
    n_a = n // 2; n_b = n - n_a
    theta_a = math.pi * torch.rand(n_a, generator=g)
    theta_b = math.pi * torch.rand(n_b, generator=g)
    A = torch.stack([torch.cos(theta_a),       torch.sin(theta_a)],       dim=1)
    B = torch.stack([1 - torch.cos(theta_b),   0.5 - torch.sin(theta_b)], dim=1)
    X = torch.cat([A, B]) + noise * torch.randn(n, 2, generator=g)
    y = torch.cat([torch.zeros(n_a), torch.ones(n_b)])
    perm = torch.randperm(n, generator=g)
    return X[perm], y[perm].unsqueeze(1)

torch.manual_seed(0)
X_all, y_all = make_moons(n=400, noise=0.15)
split = 300
X_train, y_train = X_all[:split], y_all[:split]
X_test,  y_test  = X_all[split:], y_all[split:]

plt.scatter(X_train[y_train.squeeze()==0, 0], X_train[y_train.squeeze()==0, 1], s=8, label='class 0')
plt.scatter(X_train[y_train.squeeze()==1, 0], X_train[y_train.squeeze()==1, 1], s=8, label='class 1')
plt.legend(); plt.axis('equal'); plt.title('moons train'); plt.show()

### 模型 = `nn.Sequential` 拼 Linear + 激活

用 `BCEWithLogitsLoss`，模型最后一层只输出 logits，**不要再叠 sigmoid**——和 chap3 的 logistic 回归一样。

In [ ]:
torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 1),     # 输出 logits
)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

history = []
for epoch in range(500):
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append(loss.item())

with torch.no_grad():
    train_acc = ((torch.sigmoid(model(X_train)) >= 0.5).float() == y_train).float().mean()
    test_acc  = ((torch.sigmoid(model(X_test))  >= 0.5).float() == y_test ).float().mean()
print(f'final loss {history[-1]:.4f}  train_acc {train_acc.item():.4f}  test_acc {test_acc.item():.4f}')

### 损失曲线与决策边界

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history); ax1.set_xlabel('epoch'); ax1.set_ylabel('BCE loss'); ax1.grid(alpha=0.3)

xx, yy = torch.meshgrid(
    torch.linspace(X_all[:, 0].min()-0.3, X_all[:, 0].max()+0.3, 200),
    torch.linspace(X_all[:, 1].min()-0.3, X_all[:, 1].max()+0.3, 200),
    indexing='xy',
)
grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
with torch.no_grad():
    prob = torch.sigmoid(model(grid)).reshape(xx.shape)
ax2.contourf(xx.numpy(), yy.numpy(), prob.numpy(), levels=20, alpha=0.5, cmap='coolwarm')
ax2.scatter(X_all[y_all.squeeze()==0, 0], X_all[y_all.squeeze()==0, 1], s=8, c='blue')
ax2.scatter(X_all[y_all.squeeze()==1, 0], X_all[y_all.squeeze()==1, 1], s=8, c='red')
ax2.set_title('decision boundary')
plt.tight_layout(); plt.show()

## 4. 隐藏层尺寸 / 激活函数对比

做一个小实验：固定其他设置，改隐藏单元数 + 激活函数，看测试准确率。

In [ ]:
def fit_mlp(hidden, activation, epochs=400, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(2, hidden), activation(), nn.Linear(hidden, 1))
    opt = optim.Adam(model.parameters(), lr=0.05)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        loss = loss_fn(model(X_train), y_train)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        acc = ((torch.sigmoid(model(X_test)) >= 0.5).float() == y_test).float().mean().item()
    return acc

print(f'{"hidden":>8s}  {"Tanh":>8s}  {"ReLU":>8s}  {"Sigmoid":>8s}')
for h in [2, 4, 8, 16, 32]:
    accs = {act_name: fit_mlp(h, act) for act_name, act in
            [('Tanh', nn.Tanh), ('ReLU', nn.ReLU), ('Sigmoid', nn.Sigmoid)]}
    print(f'{h:>8d}  {accs["Tanh"]:>8.4f}  {accs["ReLU"]:>8.4f}  {accs["Sigmoid"]:>8.4f}')

观察：

- 隐藏单元少（h=2）时容量不足，准确率低。
- 隐藏单元到 4-8 之后基本饱和，moons 比较简单。
- Sigmoid 在更深 / 更窄的网络里收敛慢（梯度消失），这里两层 + Adam 影响有限；后续 chap7 会进一步讨论。

## 小结

- PyTorch 的训练循环固定模板：`logits = model(x); loss = loss_fn(logits, y); opt.zero_grad(); loss.backward(); opt.step()`。
- **logits-only 约定**：分类时模型最后一层只输出 logits，`BCEWithLogitsLoss` / `CrossEntropyLoss` 内部融合 sigmoid/softmax + 交叉熵，数值更稳。
- **autograd** 通过链式法则自动反传梯度——本节通过手算验证了 PyTorch 算出的梯度与公式一致。
- `nn.Sequential` 是最轻量的网络组装方式；更复杂的结构用 `nn.Module` 子类（chap4 下示范）。